# ChakraModel YOLO Cross-Dataset Training (Optimized for Attached Dataset)

This notebook trains a YOLOv8 object detector using the `chakramodel-evaluation-datasets` you directly attached to Kaggle.

**Instructions:**
1. Ensure **GPU T4 x2** (or P100) is selected as the accelerator.
2. Run all cells below.

In [ ]:
!pip install ultralytics opencv-python-headless -q

In [ ]:
import os
import cv2
import shutil
import numpy as np
from pathlib import Path
from ultralytics import YOLO

out_dataset_dir = Path("/kaggle/working/polyp_dataset")
out_images_dir = out_dataset_dir / "images"
out_labels_dir = out_dataset_dir / "labels"

# Create YOLO directory structure
for split in ['train', 'val']:
    (out_images_dir / split).mkdir(parents=True, exist_ok=True)
    (out_labels_dir / split).mkdir(parents=True, exist_ok=True)

### Step 1: Discover Attached Datasets

In [ ]:
# Try to find the dataset root dynamically
possible_paths = [
    Path("/kaggle/input/chakramodel-evaluation-datasets"),
    Path("/kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets")
]

dataset_root = None
for p in possible_paths:
    if p.exists() and list(p.glob("*")):
        dataset_root = p
        break

if dataset_root is None:
    # Fallback search if path is slightly different
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'kvasir-seg' in dirs or 'cvc-clinicdb' in dirs:
            dataset_root = Path(root)
            break

print(f"Using dataset root: {dataset_root}")

### Step 2: Convert Masks to YOLO Bounding Boxes

In [ ]:
def mask_to_yolo(mask_path, img_w, img_h):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None: return []
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bboxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w > 0 and h > 0:
            cx = (x + w/2) / img_w
            cy = (y + h/2) / img_h
            nw = w / img_w
            nh = h / img_h
            bboxes.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
    return bboxes

all_data = []

for dataset_folder in dataset_root.iterdir():
    if not dataset_folder.is_dir(): continue
    images_dir = dataset_folder / "images"
    masks_dir = dataset_folder / "masks"
    
    if images_dir.exists() and masks_dir.exists():
        for img_p in images_dir.glob("*"):
            if img_p.suffix.lower() in ['.jpg', '.png', '.jpeg']:
                # Handle case where mask has different extension than image (e.g. img.jpg vs mask.png)
                mask_matches = list(masks_dir.glob(f"{img_p.stem}.*"))
                if mask_matches:
                    all_data.append((img_p, mask_matches[0]))

print(f"Found {len(all_data)} images with masks across all datasets.")

In [ ]:
np.random.seed(42)
np.random.shuffle(all_data)

train_split = int(0.85 * len(all_data))

for i, (img_p, msk_p) in enumerate(all_data):
    split = "train" if i < train_split else "val"
    
    img = cv2.imread(str(img_p))
    if img is None: continue
    h, w = img.shape[:2]
    
    yolo_lines = mask_to_yolo(msk_p, w, h)
    if not yolo_lines: continue
    
    new_img_name = f"{img_p.stem}_{i}.jpg"
    new_txt_name = f"{img_p.stem}_{i}.txt"
    
    # Copy image to YOLO directory
    shutil.copy2(img_p, out_images_dir / split / new_img_name)
    
    # Write YOLO label
    with open(out_labels_dir / split / new_txt_name, "w") as f:
        f.write("\n".join(yolo_lines))

print("YOLO Dataset generated successfully in /kaggle/working/polyp_dataset!")

In [ ]:
yaml_content = f"""path: {out_dataset_dir}
train: images/train
val: images/val

names:
  0: polyp
"""
with open(out_dataset_dir / "dataset.yaml", "w") as f:
    f.write(yaml_content)

### Step 3: Train YOLOv8 Model (Multi-GPU)

In [ ]:
model = YOLO("yolov8n.pt") # Using nano for speed

# Uses device=[0, 1] to train on 2 GPUs in parallel
results = model.train(
    data=str(out_dataset_dir / "dataset.yaml"),
    epochs=50,
    imgsz=640,
    batch=32, # Batch size for dual GPU
    device=[0, 1], # Utilizes both GPUs
    project="/kaggle/working/yolo_output",
    name="cross_dataset_polyp"
)

print("Training Complete! Download the weights from /kaggle/working/yolo_output/cross_dataset_polyp/weights/best.pt")